# Projet Leyenda - Livrable 3
##### *Par Noah.V, Noah.M, Hugo.L*

## **Contexte:**
L'entreprise TouNum est une entreprise qui travaille sur la numérisation de différents type de documents. Ils essayent de moderniser leur architecture pour le tri automatique de documents due au grand nombre de documents fourni par les clients. Faisant appel à nous pour determiner le captionning des images avec légendes de manière automatiques. Nous remarquons deux contraintes, la première est la qualité des images qui diffère par exemple entre chaque image, et trié automatiquement si c'est une photo ou d'autres types de documents.

Le workflow demandé par l'entreprise devra suivre ce modèle la:

![Image demandé](src/workflow.png)

Le projet sera divisé en trois partie:
- La première sera de contrsuire un CNN via des datasets d'images pré étiqueté
- Le deuxième sera d'améliorer la clarté et l'enlever les différents bruits ou flou des images
- Le troisème sera de créer un réseau de neurones qui génère des légendes pour des photographies

Après avoir réalisé les deux premiers livrables, l'objectif est de créer un réseau de neurones qui génère des légendes pour des photographies, en s'appuyant sur le dataset dataset MS COCO. Il sera divisé en 2 partie la premiere via un CNN et la deuxieme sera le RNN pour le captionning. Nous utilisons un modèle pré-entrainé: *Inception V3*

Détail du ***Livrable 3:***
- L'architecture schématique complète du réseau utilisé pour le captioning explicitant le type de CNN utilisé pour les prétraitements.
- Un petit descriptif sur le pré-traitements de images et du texte.
- Le code explicitant l'architecture du CNN et du RNN utilisés dans le captioning.
- L'évolution sous forme de courbes des performances du réseau pendant l'entrainement. Affichage de quelques exemples pour les tests.

## Importation des librairies

In [ ]:
%pip requirements.txt

In [1]:
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input


## Etape 1: Architecture schématique du modèle.

Voici l'architecture globale du modèle de captioning :
![Image demandé](src/Captioning_model.jpg)

Fonctionnement du modèle :

Le modèle prend en entrée une image et sa légende associée. L’image est prétraitée, puis transmise à InceptionV3, utilisé avec des poids gelés pour extraire ses caractéristiques visuelles. Le CNN Encoder transforme ensuite ces caractéristiques en 64 vecteurs de 256 valeurs. En parallèle, la légende est préparée, convertie en tokens et complétée par du padding.

À chaque étape, le mécanisme d’attention utilise les caractéristiques de l’image et l’état caché précédent du décodeur pour calculer un contexte visuel. Le décodeur, basé sur un LSTM ou un GRU, utilise ce contexte et le token précédent pour produire les scores du prochain token. Son nouvel état caché est conservé pour l’étape suivante.

Pendant l’entraînement, les scores sont comparés au token cible de la légende de référence afin de calculer la loss et d’ajuster les paramètres entraînables du modèle. Le processus se répète sur les différents tokens de la légende.

Voici le détail de l’architecture de l’encodeur :

![Image demandé](src/Captioning_model-CNN.jpg)

Fonctionnement du CNN Encoder :

Le CNN Encoder reçoit les caractéristiques extraites par InceptionV3, sous la forme de 64 vecteurs de 2 048 valeurs. Chaque vecteur correspond à une position dans la grille de l’image. La couche Dense applique la même projection à chacun de ces vecteurs pour passer de 2 048 à 256 valeurs. L’activation ReLU remplace ensuite les valeurs négatives par zéro. On obtient donc 64 vecteurs de 256 valeurs : la représentation est réduite, mais les différentes positions de l’image sont conservées pour le mécanisme d’attention.

Voici le détail de l’architecture du décodeur :

![Image demandé](src/Captioning_model-RNN.jpg)

Fonctionnement du RNN Decoder (GRU):


Le décodeur reçoit les caractéristiques visuelles encodées, le token précédent et l’état caché de l’étape précédente. Le token est transformé par la couche Embedding en un vecteur de 256 valeurs. En parallèle, l’attention de Bahdanau utilise les caractéristiques de l’image et l’état caché précédent pour pondérer les 64 positions et construire un contexte visuel de 256 valeurs. Ces deux vecteurs sont concaténés pour former l’entrée du GRU, soit 512 valeurs. 

Le GRU produit une sortie et un nouvel état caché de 512 valeurs, réutilisé par l’attention au tour suivant. Dans cette implémentation, l’état précédent est transmis à l’attention, mais pas directement au GRU comme état initial. La sortie passe ensuite par une couche Dense de 512 neurones, une activation ReLU et un Reshape qui adapte sa forme avant la dernière couche. 

La couche Dense finale produit 5 001 scores, appelés logits, soit un par identifiant du vocabulaire de sortie. Pendant l’entraînement, ces scores sont comparés au token cible pour calculer la loss. Lors de la génération, le code sélectionne le prochain token par échantillonnage à partir des logits, puis le réutilise comme entrée à l’étape suivante, jusqu’au token de fin ou à la longueur maximale autorisée.


## Etape 2 : Chargement et Pré-traitement des images


### Importation du dataset et annotation COCO

In [2]:
DATA_DIR = Path("Data")

ANNOTATION_FILES = {
    "train": DATA_DIR / "annotations" / "captions_train2014.json",
    "validation": DATA_DIR / "annotations" / "captions_val2014.json",
}

IMAGE_DIRS = {
    "train": DATA_DIR / "train2014",
    "validation": DATA_DIR / "val2014",
}

def load_split(split):
    with ANNOTATION_FILES[split].open("r", encoding="utf-8") as f:
        data = json.load(f)
    
    records = {}
    for image in data["images"]:
        image_id = image["id"]
        if image_id in records:
            raise ValueError(f"{split} : ID image dupliqué {image_id}")
        records[image_id] = {
            "image_id": image_id,
            "image_path": IMAGE_DIRS[split] / image["file_name"],
            "captions": [],
        }

    for ann in data["annotations"]:
        image_id = ann["image_id"]
        if image_id not in records:
            raise ValueError(f"{split} : légende liée à une image inconnue {image_id}")
        records[image_id]["captions"].append(ann["caption"])
 
    missing_files = [r["image_path"] for r in records.values() if not r["image_path"].is_file()]
    no_caption = [i for i, r in records.items() if not r["captions"]]
    n_captions = sum(len(r["captions"]) for r in records.values())

    print(f"{split} : {len(records):,} images, {n_captions:,} légendes")
    print(f"  images introuvables sur disque : {len(missing_files)}")
    print(f"  images sans légende            : {len(no_caption)}")

    if missing_files:
        print("  exemples manquants :", missing_files[:3])

    return records

train_records = load_split("train")
val_records = load_split("validation")

shared = set(train_records) & set(val_records)
print("IDs communs train/validation :", len(shared))

example = next(iter(train_records.values()))

print("ID image :", example["image_id"])
print("Chemin :", example["image_path"])

for caption in example["captions"]:
    print("-", caption)

train : 82,783 images, 414,113 légendes
  images introuvables sur disque : 0
  images sans légende            : 0
validation : 40,504 images, 202,654 légendes
  images introuvables sur disque : 0
  images sans légende            : 0
IDs communs train/validation : 0
ID image : 57870
Chemin : Data\train2014\COCO_train2014_000000057870.jpg
- A restaurant has modern wooden tables and chairs.
- A long restaurant table with rattan rounded back chairs.
- a long table with a plant on top of it surrounded with wooden chairs 
- A long table with a flower arrangement in the middle for meetings
- A table is adorned with wooden chairs with blue accents.


Dans cette partie, nous chargeons les deux dossiers **train2014** et **val2014**, qui se trouvent dans le dossier principal **Data**. Nous effectuons ensuite une vérification pour nous assurer qu'il y a le même nombre d'images et d'annotations, afin de garantir l'intégrité du chargement et d'éviter toute erreur.

Après cela, nous prenons un exemple d'image avec ses légendes associées, afin de vérifier que la cellule a bien relié chaque image à ses légendes.

Ensuite, nous commencons par le pré-proccessing des images.

In [3]:
IMAGE_SIZE = (299, 299)
BATCH_SIZE = 32

inception = InceptionV3(
    weights="imagenet",
    include_top=False,
    pooling=None,
    input_shape=(*IMAGE_SIZE, 3),
)

inception.trainable = False
inputs = Input(
    shape=(*IMAGE_SIZE, 3),
    name="image_input",
)

outputs = inception(inputs, training=False)

image_encoder = Model(
    inputs=inputs,
    outputs=outputs,
    name="image_encoder",
)

print("Entrée :", image_encoder.input_shape)
print("Sortie :", image_encoder.output_shape)

Entrée : (None, 299, 299, 3)
Sortie : (None, 8, 8, 2048)


Dans un premier temps, nous utilisons le modèle InceptionV3 préentraîné pour extraire les features des images. Ces caractéristiques sont ensuite transmises à notre encodeur CNN. Nous nous appuyons ainsi sur le transfert learning afin de réutiliser les connaissances acquises par le modèle lors de son préentraînement.

Nous retirons la tête de classification d’InceptionV3 en définissant le paramètre *include_top=False*. Cela permet de conserver la partie du réseau dédiée à l’extraction des caractéristiques, sans utiliser sa couche finale de classification. Les autres paramètres précisent notamment les poids préentraînés à charger et l’absence de pooling supplémentaire en sortie.

Enfin, nous créons un modèle d’extraction en définissant explicitement son entrée et sa sortie à partir du réseau InceptionV3.

In [ ]:
def load_and_preprocess_image(image_id, image_path):
    image_bytes = tf.io.read_file(image_path)

    image = tf.io.decode_jpeg(
        image_bytes,
        channels=3,
    )

    image = tf.image.resize(image, IMAGE_SIZE)
    image = preprocess_input(image)

    return image_id, image

In [ ]:
def make_image_dataset(records):
    image_ids = np.array(list(records.keys()), dtype=np.int64)

    image_paths = [
        str(records[int(image_id)]["image_path"])
        for image_id in image_ids
    ]

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_ids, image_paths)
    )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

train_image_dataset = make_image_dataset(train_records)
val_image_dataset = make_image_dataset(val_records)

print("Datasets train et validation créés.")

Ici, nous définissons d’abord la fonction *load_and_preprocess_image()*, qui charge une image depuis son chemin, la décode en trois canaux RGB et la redimensionne à une taille de 299 × 299 pixels. Elle applique ensuite le prétraitement adapté à InceptionV3, afin de préparer l’image à l’extraction de ses caractéristiques. La fonction renvoie l’identifiant de l’image ainsi que l’image prétraitée.

Nous créons ensuite la fonction *make_image_dataset()*, qui construit un dataset à partir des identifiants des images et de leurs chemins d’accès. Elle applique la fonction de prétraitement à chaque image, regroupe les résultats en batches et utilise le préchargement pour préparer les batches suivants pendant le traitement du batch courant. Les identifiants restent associés aux images, ce qui permettra ensuite d’enregistrer les caractéristiques extraites dans les fichiers correspondants.

In [ ]:
FEATURES_DIR = Path("features/inception_grid_fp16")

N_TRAIN_IMAGES = 30_000
N_VAL_IMAGES = 10_000
SEED = 42

def extract_features(records, split_name, n_images=None):
    out_dir = FEATURES_DIR / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
   
    if n_images is not None:
        all_ids = np.array(sorted(records), dtype=np.int64)
        rng = np.random.default_rng(SEED)
        selected = rng.choice(all_ids, size=min(n_images, len(all_ids)), replace=False)
        records = {int(i): records[int(i)] for i in selected}

    pending = [i for i in records if not (out_dir / f"{i}.npy").exists()]
    already_done = len(records) - len(pending)

    print(f"{split_name} : {len(records):,} images retenues, "
          f"{already_done:,} déjà faites, {len(pending):,} à traiter")
    if not pending:
        return

    dataset = make_image_dataset({i: records[i] for i in pending})
    total_batches = (len(pending) + BATCH_SIZE - 1) // BATCH_SIZE

    start = time.time()
    done = 0

    for batch_number, (image_ids, images) in enumerate(dataset, start=1):
        features = image_encoder(images, training=False)                      
        features = tf.reshape(features, (tf.shape(features)[0], 64, 2048))   
        features = features.numpy().astype(np.float16)

        for image_id, vector in zip(image_ids.numpy(), features):
            tmp_path = out_dir / f"{image_id}.tmp.npy"
            np.save(tmp_path, vector)
            tmp_path.replace(out_dir / f"{image_id}.npy")

        done += len(features)

        if batch_number % 100 == 0 or batch_number == total_batches:
            speed = done / (time.time() - start)
            print(f"{split_name} : {batch_number}/{total_batches} batches, {speed:.1f} images/s", flush=True)
extract_features(train_records, "train", n_images=N_TRAIN_IMAGES)
extract_features(val_records, "validation", n_images=N_VAL_IMAGES)

Dans cette partie, nous utilisons InceptionV3 pour extraire les features des images prétraitées, puis nous les enregistrons dans des fichiers au format .npy. Chaque fichier est nommé à partir de l’identifiant de l’image correspondante, afin de conserver cette association. Cette extraction préalable permet de réutiliser les caractéristiques pendant l’entraînement sans recalculer la sortie d’InceptionV3 à chaque passage sur les images, ce qui réduit le temps et les ressources de calcul nécessaires.

In [ ]:
def prepare_captions(records):
    image_ids = []
    captions = []
    for image_id, record in records.items():
        for caption in record["captions"]:
            image_ids.append(image_id)
            captions.append(f"startseq {caption.strip()} endseq")

    return np.asarray(image_ids, dtype=np.int64), captions

train_caption_image_ids, train_captions = prepare_captions(train_records)
val_caption_image_ids, val_captions = prepare_captions(val_records)

print("Légendes train :", len(train_captions))
print("Légendes validation :", len(val_captions))

Ensuite, la fonction *prepare_captions()* prépare deux listes alignées : l’une contenant les identifiants des images et l’autre leurs légendes. Lorsqu’une image possède plusieurs légendes, son identifiant est répété pour chacune d’elles. Cette organisation permettra ensuite d’associer chaque légende aux caractéristiques de l’image correspondante. La fonction ajoute également les marqueurs *startseq* et *endseq* autour de chaque légende pour en indiquer le début et la fin.

In [ ]:
TOP_K = 5000

tokenizer = Tokenizer(num_words=TOP_K, oov_token="unk")
tokenizer.fit_on_texts(train_captions)   

train_sequences = tokenizer.texts_to_sequences(train_captions)
val_sequences = tokenizer.texts_to_sequences(val_captions)

vocab_size = min(TOP_K, len(tokenizer.word_index))

start_token_id = tokenizer.word_index["startseq"]
end_token_id = tokenizer.word_index["endseq"]

assert start_token_id < vocab_size
assert end_token_id < vocab_size

print("vocab_size :", vocab_size)
print("startseq :", start_token_id, "| endseq :", end_token_id)

max_length = 20 # Ancien max(len(sequence) for sequence in train_sequences)

val_sequences = [
    sequence
    if len(sequence) <= max_length
    else sequence[:max_length - 1] + [end_token_id]
    for sequence in val_sequences
]

train_caption_tokens = pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post",
    value=0,
    dtype="int32",
)

val_caption_tokens = pad_sequences(
    val_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post",
    value=0,
    dtype="int32",
)

print("Longueur maximale :", max_length)
print("Train :", train_caption_tokens.shape)
print("Validation :", val_caption_tokens.shape)

print("vocab_size :", vocab_size)
print("max_length :", max_length)
print("indice max dans train :", train_caption_tokens.max())
print("indice max dans validation :", val_caption_tokens.max())
print("unk :", tokenizer.word_index["unk"],
      "| startseq :", start_token_id,
      "| endseq :", end_token_id)

assert train_caption_tokens.max() < vocab_size
assert val_caption_tokens.max() < vocab_size

Ici, nous utilisons le tokenizer pour construire un vocabulaire à partir des légendes d’entraînement et associer un identifiant numérique à chaque mot retenu. Les légendes d’entraînement et de validation sont ensuite converties en séquences d’identifiants à l’aide de ce même vocabulaire. Ces séquences serviront d’entrée à la couche d’embedding du décodeur, qui transformera chaque identifiant en un vecteur appris pendant l’entraînement. 

Le paramètre TOP_K = 5000 limite le vocabulaire utilisé lors de la conversion des légendes aux mots les plus fréquents du corpus d’entraînement. Les mots absents du vocabulaire ou dont l’identifiant atteint ou dépasse cette limite sont remplacés par le token unk (unknown). Cela permet de limiter la taille du vocabulaire et celle de la couche de sortie du décodeur.

## Etape 3: Architecture avec CNN et RNN

In [ ]:
EMBEDDING_DIM = 256   

class CNN_Encoder(tf.keras.Model):
    def __init__(self, embedding_dim):
        super().__init__()
        self.fc = tf.keras.layers.Dense(embedding_dim)

    def call(self, x):
        x = self.fc(x)        
        x = tf.nn.relu(x)
        return x


encoder = CNN_Encoder(EMBEDDING_DIM)

2000


Ici, nous créons notre encodeur CNN, qui transforme les features extraites par InceptionV3 en vecteurs de 256 dimensions grâce à une couche Dense suivie d’une activation ReLU, afin de les préparer pour le mécanisme d’attention et le décodeur.

In [ ]:
#Mécanisme d'attention
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, features, hidden):
        hidden_with_time_axis = tf.expand_dims(hidden, axis=1)
        
        attention_hidden = tf.nn.tanh(
            self.W1(features) + self.W2(hidden_with_time_axis)
        )
        scores = self.V(attention_hidden)
        attention_weights = tf.nn.softmax(scores, axis=1)
        context_vector = tf.reduce_sum(
            attention_weights * features,
            axis=1,
        )

        return context_vector, attention_weights

RNN_UNITS = 512
ATTENTION_UNITS = 512

attention_test = BahdanauAttention(ATTENTION_UNITS)
encoded_features = encoder(batch_features)

hidden_test = tf.zeros(
    (tf.shape(encoded_features)[0], RNN_UNITS),
    dtype=encoded_features.dtype,
)

context_vector, attention_weights = attention_test(
    encoded_features,
    hidden_test,
)

print("Caractéristiques encodées :", encoded_features.shape)
print("État caché :", hidden_test.shape)
print("Vecteur de contexte :", context_vector.shape)
print("Poids d’attention :", attention_weights.shape)

tf.debugging.assert_near(
    tf.reduce_sum(attention_weights, axis=1),
    tf.ones_like(attention_weights[:, 0, :]),
    atol=1e-5,)

tf.debugging.assert_all_finite(
    context_vector,
    "Le vecteur de contexte contient des valeurs invalides.",
)

Ici, nous créons le mécanisme d’attention de Bahdanau, qui utilise les features de l’image et l’état caché du décodeur pour donner plus d’importance aux zones utiles à la prédiction du prochain mot. Il produit un vecteur de contexte résumant ces informations, puis nous vérifions sur un batch que les dimensions et les poids d’attention sont cohérents.

In [ ]:
class RNN_Decoder(tf.keras.Model):
    def __init__(self, embedding_dim, units, vocab_size):
        super().__init__()
        self.units = units
        self.embedding = layers.Embedding(vocab_size, embedding_dim)
        self.attention = BahdanauAttention(units)
        self.lstm = layers.LSTM(units, return_sequences=True, return_state=True)
        self.fc1 = layers.Dense(units)
        self.fc2 = layers.Dense(vocab_size)

    def call(self, word_id, features, hidden, cell):
        # contexte visuel selon l'état caché courant
        context_vector, attention_weights = self.attention(features, hidden)

        x = self.embedding(word_id)                                
        context = tf.expand_dims(context_vector, axis=1)           
        x = tf.concat([context, x], axis=-1)                       

        output, hidden, cell = self.lstm(x, initial_state=[hidden, cell])

        x = self.fc1(output)                                       
        x = tf.reshape(x, (-1, x.shape[2]))                        
        logits = self.fc2(x)                                       

        return logits, hidden, cell, attention_weights

    def reset_state(self, batch_size):
        zeros = tf.zeros((batch_size, self.units))
        return zeros, zeros


decoder = RNN_Decoder(EMBEDDING_DIM, RNN_UNITS, vocab_size)

Dans notre architecture, le décodeur RNN utilise un LSTM pour générer la légende mot par mot. Il transforme le mot précédent en vecteur grâce à l’embedding, puis le combine au contexte visuel fourni par l’attention. Le LSTM utilise ces informations et ses états précédents pour produire, à travers deux couches Dense, un score pour chaque mot du vocabulaire afin de prédire le mot suivant.